# PhysicalAI-AV Clip 下载与可视化

本 notebook 使用 `physical-ai-av` 库完成指定 `CLIP_ID` 的数据读取、下载和可视化。
通过 `MAYBE_STREAM` 统一控制两种数据访问方案：

- `True`：按需流式读取，不预下载完整 feature；
- `False`：先将所需 feature 下载到 `CACHE_DIR`，后续只读本地缓存。


In [ ]:
import os, io, json, textwrap, zipfile, warnings
from pathlib import Path

os.environ['HF_ENDPOINT'] = 'https://huggingface.co'
os.environ.setdefault('HF_HUB_DISABLE_XET', '1')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation, font_manager
from matplotlib.patches import Rectangle

import physical_ai_av
from physical_ai_av import PhysicalAIAVDatasetInterface

warnings.filterwarnings('ignore')

CACHE_DIR = Path(globals().get(
    'CACHE_DIR',
    'PhysicalAI-Autonomous-Vehicles/cache_clips',
))
MAYBE_STREAM = bool(globals().get('MAYBE_STREAM', True))
REASONING_FILE = 'reasoning/ood_reasoning.parquet'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CJK_FONT_CANDIDATES = [
    '/usr/share/fonts/truetype/wqy/wqy-microhei.ttc',
    '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
]
CJK_FONT_PATH = next((p for p in CJK_FONT_CANDIDATES if Path(p).exists()), None)
if CJK_FONT_PATH:
    font_manager.fontManager.addfont(CJK_FONT_PATH)
    CJK_FONT_PROP = font_manager.FontProperties(fname=CJK_FONT_PATH)
    CJK_FONT_NAME = CJK_FONT_PROP.get_name()
else:
    CJK_FONT_PROP = font_manager.FontProperties(family='DejaVu Sans')
    CJK_FONT_NAME = 'DejaVu Sans'

plt.rcParams['font.sans-serif'] = [CJK_FONT_NAME, 'WenQuanYi Micro Hei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = '#12121a'
plt.rcParams['axes.facecolor'] = '#1a1a24'
plt.rcParams['axes.edgecolor'] = '#333'
plt.rcParams['axes.labelcolor'] = '#f0f0f5'
plt.rcParams['axes.titlecolor'] = '#f0f0f5'
plt.rcParams['text.color'] = '#f0f0f5'
plt.rcParams['xtick.color'] = '#aaa'
plt.rcParams['ytick.color'] = '#aaa'
PALETTE = ['#6366f1', '#8b5cf6', '#06b6d4', '#10b981', '#f59e0b', '#ef4444']

def apply_cjk_font(fig):
    for text in fig.findobj(match=lambda x: hasattr(x, 'set_fontproperties')):
        try:
            text.set_fontproperties(CJK_FONT_PROP)
        except Exception:
            pass
    if getattr(fig, '_suptitle', None) is not None:
        fig._suptitle.set_fontproperties(CJK_FONT_PROP)
    return fig

print('physical_ai_av:', physical_ai_av.__version__)
print('cache:', CACHE_DIR)
print('data mode:', 'streaming' if MAYBE_STREAM else 'local cache (eager download)')
print('font:', CJK_FONT_NAME)


## 1. 初始化数据集接口

如需登录，请先在终端执行：

```bash
HF_ENDPOINT=https://huggingface.co hf auth login
```

并确认已接受 `nvidia/PhysicalAI-Autonomous-Vehicles` gated dataset 协议。


In [ ]:
avdi = PhysicalAIAVDatasetInterface(
    cache_dir=CACHE_DIR,
    token=True,
    confirm_download_threshold_gb=float('inf'),
)
print('clip_index:', avdi.clip_index.shape)
print('feature_presence:', avdi.feature_presence.shape)
print('revision:', avdi.revision)
avdi.clip_index.head()


## 2. 指定 clip_id 与数据访问方案

默认使用 `0f8865ed-8fba-4d37-82c4-c3c763e9f325`。可以在运行本单元格前覆盖：

```python
CLIP_ID = 'your-clip-id'
MAYBE_STREAM = True   # 流式按需读取
# MAYBE_STREAM = False  # 完整下载到 CACHE_DIR 后读取
```


In [ ]:
CLIP_ID = str(globals().get('CLIP_ID', '0f8865ed-8fba-4d37-82c4-c3c763e9f325'))

CAMERA_FEATURES = [
    avdi.features.CAMERA.CAMERA_CROSS_LEFT_120FOV,
    avdi.features.CAMERA.CAMERA_FRONT_WIDE_120FOV,
    avdi.features.CAMERA.CAMERA_CROSS_RIGHT_120FOV,
    avdi.features.CAMERA.CAMERA_FRONT_TELE_30FOV,
]
DOWNLOAD_FEATURES = [
    avdi.features.LABELS.EGOMOTION,
    avdi.features.LABELS.OBSTACLE_OFFLINE,
    avdi.features.CALIBRATION.CAMERA_INTRINSICS,
    avdi.features.CALIBRATION.SENSOR_EXTRINSICS,
    avdi.features.CALIBRATION.VEHICLE_DIMENSIONS,
    *CAMERA_FEATURES,
]

if CLIP_ID not in avdi.clip_index.index:
    raise ValueError(f'clip_id 不存在: {CLIP_ID}')
CHUNK_ID = int(avdi.get_clip_chunk(CLIP_ID))
print('CLIP_ID:', CLIP_ID)
print('chunk:', CHUNK_ID)
print('split:', avdi.clip_index.loc[CLIP_ID, 'split'])
print('data mode:', 'streaming' if MAYBE_STREAM else 'local cache (eager download)')
for feature in DOWNLOAD_FEATURES:
    print('  -', feature)

if MAYBE_STREAM:
    print('流式模式：跳过预下载，后续 get/open 调用将按需访问远端或已有缓存。')
else:
    avdi.download_clip_features(
        CLIP_ID,
        features=DOWNLOAD_FEATURES,
        max_workers=2,
        maybe_stream=False,
        show_progress=True,
    )
    print('非流式模式：下载完成 / 已命中 cache。')
    try:
        avdi.download_files([REASONING_FILE], max_workers=1)
        print('CoC 标注索引下载完成 / 已命中 cache。')
    except Exception as exc:
        print(f'提示: 可选 CoC 标注索引不可用，将跳过 CoT 覆盖层。{exc}')


## 3. 加载 physical-ai-av 数据

In [ ]:
egomotion_interp = avdi.get_clip_feature(CLIP_ID, avdi.features.LABELS.EGOMOTION, maybe_stream=MAYBE_STREAM)
vehicle_dims = avdi.get_clip_feature(CLIP_ID, avdi.features.CALIBRATION.VEHICLE_DIMENSIONS, maybe_stream=MAYBE_STREAM)
camera_intrinsics = avdi.get_clip_feature(CLIP_ID, avdi.features.CALIBRATION.CAMERA_INTRINSICS, maybe_stream=MAYBE_STREAM)
sensor_extrinsics = avdi.get_clip_feature(CLIP_ID, avdi.features.CALIBRATION.SENSOR_EXTRINSICS, maybe_stream=MAYBE_STREAM)

# obstacle.offline 在 chunk 级索引中可能存在，但个别 clip 的 parquet 可能未打包进 ZIP。
# 将其视为可选 feature，避免阻断 egomotion、相机和综合 GIF 流程。
obstacle_data = None
obstacle_load_error = None
try:
    obstacle_data = avdi.get_clip_feature(
        CLIP_ID,
        avdi.features.LABELS.OBSTACLE_OFFLINE,
        maybe_stream=MAYBE_STREAM,
    )
except KeyError as exc:
    obstacle_load_error = str(exc)
    print(f'提示: 当前 clip 缺少 obstacle.offline 文件，跳过障碍物可视化。{exc}')

# CoC 是仓库级可选标注；模型接口中对应的输出字段通常名为 cot。
coc_events = []
reasoning_load_error = None
try:
    with avdi.open_file(REASONING_FILE, maybe_stream=MAYBE_STREAM) as reasoning_file:
        reasoning_df = pd.read_parquet(reasoning_file)

    if isinstance(reasoning_df.index, pd.MultiIndex) and 'clip_id' in reasoning_df.index.names:
        clip_reasoning = reasoning_df[
            reasoning_df.index.get_level_values('clip_id').astype(str) == CLIP_ID
        ]
    elif reasoning_df.index.name == 'clip_id':
        clip_reasoning = reasoning_df.loc[[CLIP_ID]] if CLIP_ID in reasoning_df.index else reasoning_df.iloc[0:0]
    elif 'clip_id' in reasoning_df.columns:
        clip_reasoning = reasoning_df[reasoning_df['clip_id'].astype(str) == CLIP_ID]
    else:
        clip_reasoning = reasoning_df.iloc[0:0]

    for _, reasoning_row in clip_reasoning.iterrows():
        events = reasoning_row.get('events', [])
        if isinstance(events, str):
            events = json.loads(events)
        if isinstance(events, list):
            coc_events.extend(
                dict(event) for event in events
                if isinstance(event, dict) and str(event.get('coc', '')).strip()
            )
    coc_events.sort(key=lambda event: int(event.get('event_start_timestamp', 0)))
except (FileNotFoundError, KeyError, ValueError, json.JSONDecodeError) as exc:
    reasoning_load_error = str(exc)
    print(f'提示: CoC 标注读取失败，将跳过 CoT 覆盖层。{exc}')

print('CoC events:', len(coc_events))
print('vehicle_dims:', vehicle_dims)
print('camera_intrinsics:', type(camera_intrinsics))
print('sensor_extrinsics:', type(sensor_extrinsics))
print('obstacle_data:', type(obstacle_data) if obstacle_data is not None else 'unavailable')


In [ ]:
chunk_file = avdi.features.get_chunk_feature_filename(CHUNK_ID, avdi.features.LABELS.EGOMOTION)
clip_files = avdi.features.get_clip_files_in_zip(CLIP_ID, avdi.features.LABELS.EGOMOTION)
with avdi.open_file(chunk_file, maybe_stream=MAYBE_STREAM) as f:
    with zipfile.ZipFile(f) as zf:
        ego = pd.read_parquet(io.BytesIO(zf.read(clip_files['egomotion'])))
ego['t_s'] = (ego['timestamp'] - ego['timestamp'].iloc[0]) / 1e6
ego['speed'] = np.linalg.norm(ego[['vx', 'vy', 'vz']].to_numpy(), axis=1)
ego['acceleration'] = np.gradient(ego['speed'].to_numpy(), ego['t_s'].to_numpy())
print('egomotion:', ego.shape, 'duration_s:', float(ego['t_s'].iloc[-1]))
ego.head()


## 4. 自车运动可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
x = ego['x'] - ego['x'].iloc[0]
y = ego['y'] - ego['y'].iloc[0]
sc = axes[0].scatter(x, y, c=ego['speed'], cmap='plasma', s=8)
axes[0].plot(x, y, color='#6366f1', alpha=0.4)
axes[0].scatter([0], [0], s=120, color='#06b6d4', edgecolors='white', label='起点')
axes[0].scatter([x.iloc[-1]], [y.iloc[-1]], s=140, color='#ef4444', marker='*', edgecolors='white', label='终点')
axes[0].set_aspect('equal')
axes[0].set_title('自车二维轨迹')
axes[0].set_xlabel('Δx (m)')
axes[0].set_ylabel('Δy (m)')
axes[0].legend()
cb = plt.colorbar(sc, ax=axes[0])
cb.set_label('速度 (m/s)')

axes[1].plot(ego['t_s'], ego['speed'], color='#06b6d4', lw=1.5)
axes[1].fill_between(ego['t_s'], ego['speed'], color='#06b6d4', alpha=0.25)
axes[1].set_title('速度曲线')
axes[1].set_xlabel('时间 (s)')
axes[1].set_ylabel('速度 (m/s)')
axes[1].grid(alpha=0.2)

axes[2].plot(ego['t_s'], ego['acceleration'], color='#f59e0b', lw=1.3)
axes[2].axhline(0, color='white', lw=0.8, alpha=0.4)
axes[2].fill_between(ego['t_s'], ego['acceleration'], 0, color='#f59e0b', alpha=0.2)
axes[2].set_title('加速度曲线')
axes[2].set_xlabel('时间 (s)')
axes[2].set_ylabel('加速度 (m/s²)')
axes[2].grid(alpha=0.2)
fig.suptitle(f'PhysicalAI-AV 自车运动: {CLIP_ID[:8]}…')
apply_cjk_font(fig)
plt.tight_layout()
plt.show()


## 5. 障碍物可视化

In [ ]:
obs = None
if isinstance(obstacle_data, dict):
    for v in obstacle_data.values():
        if isinstance(v, pd.DataFrame) and {'center_x', 'center_y', 'label_class'}.issubset(v.columns):
            obs = v.copy()
            break
elif isinstance(obstacle_data, pd.DataFrame):
    obs = obstacle_data.copy()

if obs is None or len(obs) == 0:
    print('当前 clip 没有可用 obstacle.offline 数据', obstacle_load_error or '')
else:
    cls_counts = obs['label_class'].value_counts()
    pivot_ts = int(obs.groupby('timestamp_us').size().sort_values(ascending=False).index[0])
    frame_obs = obs[obs['timestamp_us'] == pivot_ts]
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].bar(cls_counts.index, cls_counts.values, color=PALETTE[:len(cls_counts)])
    axes[0].set_title('障碍物类别分布')
    axes[0].set_ylabel('观测次数')
    axes[0].tick_params(axis='x', rotation=30)
    for _, r in frame_obs.iterrows():
        axes[1].scatter(r['center_x'], r['center_y'], s=30, alpha=0.8)
    axes[1].add_patch(Rectangle((-2.0, -1.0), 4.8, 2.0, edgecolor='white', facecolor='#06b6d4', alpha=0.35))
    axes[1].set_aspect('equal')
    axes[1].set_xlim(-30, 90)
    axes[1].set_ylim(-40, 40)
    axes[1].set_title(f'BEV 障碍物散点 ts={pivot_ts}')
    axes[1].set_xlabel('X (m, 车前)')
    axes[1].set_ylabel('Y (m, 车左)')
    axes[1].grid(alpha=0.2)
    fig.suptitle('障碍物检测可视化')
    apply_cjk_font(fig)
    plt.tight_layout()
    plt.show()


## 6. 多相机帧可视化

In [ ]:
camera_readers = {}
for cam_feature in CAMERA_FEATURES:
    try:
        reader = avdi.get_clip_feature(CLIP_ID, cam_feature, maybe_stream=MAYBE_STREAM)
        camera_readers[cam_feature] = reader
        print('loaded:', cam_feature, 'frames=', len(reader.timestamps) if reader.timestamps is not None else 'unknown')
    except Exception as e:
        print('failed:', cam_feature, e)

ncam = max(1, len(camera_readers))
fig, axes = plt.subplots(1, ncam, figsize=(4 * ncam, 3.2))
if ncam == 1:
    axes = [axes]
for ax, (cam_feature, reader) in zip(axes, camera_readers.items()):
    n = len(reader.timestamps) if reader.timestamps is not None else 1
    idx = np.array([max(0, n // 2)], dtype=np.int64)
    img = reader.decode_images_from_frame_indices(idx)[0]
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(cam_feature.replace('camera_', '').replace('_', ' '), fontsize=10)
for ax in axes[len(camera_readers):]:
    ax.axis('off')
fig.suptitle(f'多相机中间帧: {CLIP_ID[:8]}…')
apply_cjk_font(fig)
plt.tight_layout()
plt.show()


## 7. 综合导出

除摘要 JSON 和 egomotion CSV 外，本章还导出一个时间同步的拼图 GIF：上方为多视角相机图像，下方为自车轨迹、速度曲线和加速度曲线；若当前 clip 存在 CoC 标注，还会按事件时间显示 CoT（CoC 标注）文本。可通过 `GIF_FPS` 和 `GIF_MAX_FRAMES` 控制体积与生成时间。


In [ ]:
OUTPUT_DIR = Path(globals().get('OUTPUT_DIR', CACHE_DIR / 'vis_output'))
GIF_FPS = int(globals().get('GIF_FPS', 5))
GIF_MAX_FRAMES = int(globals().get('GIF_MAX_FRAMES', 120))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not camera_readers:
    raise RuntimeError('没有可用相机 reader，无法生成多视角综合 GIF。请先运行第 6 章。')
if GIF_FPS <= 0 or GIF_MAX_FRAMES <= 0:
    raise ValueError('GIF_FPS 和 GIF_MAX_FRAMES 必须为正整数。')

ego_ts = ego['timestamp'].to_numpy(dtype=np.int64)
camera_items = list(camera_readers.items())
common_start_us = max([int(ego_ts[0])] + [int(np.asarray(r.timestamps)[0]) for _, r in camera_items])
common_end_us = min([int(ego_ts[-1])] + [int(np.asarray(r.timestamps)[-1]) for _, r in camera_items])
if common_end_us <= common_start_us:
    raise RuntimeError('自车与相机时间戳没有公共区间，无法生成同步 GIF。')

duration_s = (common_end_us - common_start_us) / 1e6
n_gif_frames = min(GIF_MAX_FRAMES, max(2, int(duration_s * GIF_FPS) + 1))
gif_timestamps = np.linspace(common_start_us, common_end_us, n_gif_frames).astype(np.int64)

def nearest_indices(sorted_timestamps, targets):
    timestamps = np.asarray(sorted_timestamps, dtype=np.int64)
    right = np.searchsorted(timestamps, targets, side='left')
    right = np.clip(right, 0, len(timestamps) - 1)
    left = np.clip(right - 1, 0, len(timestamps) - 1)
    choose_left = np.abs(targets - timestamps[left]) <= np.abs(timestamps[right] - targets)
    return np.where(choose_left, left, right).astype(np.int64)

ego_frame_indices = nearest_indices(ego_ts, gif_timestamps)
camera_frame_indices = {
    feature: nearest_indices(reader.timestamps, gif_timestamps)
    for feature, reader in camera_items
}

trajectory_x = (ego['x'] - ego['x'].iloc[0]).to_numpy()
trajectory_y = (ego['y'] - ego['y'].iloc[0]).to_numpy()
time_s = ego['t_s'].to_numpy()
speed = ego['speed'].to_numpy()
acceleration = ego['acceleration'].to_numpy()

has_coc = bool(coc_events)
figure_height = 10 if has_coc else 9
height_ratios = [1.0, 1.15, 0.28] if has_coc else [1.0, 1.15]
fig = plt.figure(figsize=(16, figure_height), constrained_layout=True)
grid = fig.add_gridspec(len(height_ratios), 6, height_ratios=height_ratios)
camera_grid = grid[0, :].subgridspec(1, len(camera_items), wspace=0.03)
camera_axes = [fig.add_subplot(camera_grid[0, i]) for i in range(len(camera_items))]
trajectory_ax = fig.add_subplot(grid[1, 0:2])
speed_ax = fig.add_subplot(grid[1, 2:4])
acceleration_ax = fig.add_subplot(grid[1, 4:6])
cot_ax = fig.add_subplot(grid[2, :]) if has_coc else None

coc_start_us = np.array(
    [int(event.get('event_start_timestamp', 0)) for event in coc_events],
    dtype=np.int64,
)
if len(coc_start_us) and coc_start_us[0] >= ego_ts[0]:
    coc_start_us = coc_start_us - ego_ts[0]

def draw_gif_frame(frame_number):
    ego_i = int(ego_frame_indices[frame_number])
    current_t = float(time_s[ego_i])

    for ax, (feature, reader) in zip(camera_axes, camera_items):
        ax.clear()
        camera_i = int(camera_frame_indices[feature][frame_number])
        image = reader.decode_images_from_frame_indices(np.array([camera_i], dtype=np.int64))[0]
        ax.imshow(image)
        ax.set_title(feature.replace('camera_', '').replace('_', ' '), fontsize=10)
        ax.axis('off')

    trajectory_ax.clear()
    trajectory_ax.plot(trajectory_x, trajectory_y, color='#4b5563', lw=1.0, alpha=0.8)
    trajectory_ax.plot(trajectory_x[:ego_i + 1], trajectory_y[:ego_i + 1], color='#8b5cf6', lw=2.0)
    trajectory_ax.scatter(trajectory_x[ego_i], trajectory_y[ego_i], s=70, color='#06b6d4', edgecolors='white', zorder=3)
    trajectory_ax.set_aspect('equal', adjustable='datalim')
    trajectory_ax.set_title('自车轨迹')
    trajectory_ax.set_xlabel('Δx (m)')
    trajectory_ax.set_ylabel('Δy (m)')
    trajectory_ax.grid(alpha=0.2)

    speed_ax.clear()
    speed_ax.plot(time_s, speed, color='#334155', lw=1.0)
    speed_ax.plot(time_s[:ego_i + 1], speed[:ego_i + 1], color='#06b6d4', lw=2.0)
    speed_ax.axvline(current_t, color='white', lw=0.9, alpha=0.7)
    speed_ax.scatter(current_t, speed[ego_i], color='#06b6d4', s=45, zorder=3)
    speed_ax.set_xlim(time_s[0], time_s[-1])
    speed_ax.set_ylim(0, max(1.0, float(np.nanmax(speed)) * 1.08))
    speed_ax.set_title(f'速度 {speed[ego_i]:.2f} m/s')
    speed_ax.set_xlabel('时间 (s)')
    speed_ax.set_ylabel('速度 (m/s)')
    speed_ax.grid(alpha=0.2)

    acceleration_ax.clear()
    acceleration_ax.plot(time_s, acceleration, color='#3f3f46', lw=1.0)
    acceleration_ax.plot(time_s[:ego_i + 1], acceleration[:ego_i + 1], color='#f59e0b', lw=1.8)
    acceleration_ax.axhline(0, color='white', lw=0.7, alpha=0.35)
    acceleration_ax.axvline(current_t, color='white', lw=0.9, alpha=0.7)
    acceleration_ax.scatter(current_t, acceleration[ego_i], color='#f59e0b', s=45, zorder=3)
    acceleration_ax.set_xlim(time_s[0], time_s[-1])
    acc_limit = max(1.0, float(np.nanpercentile(np.abs(acceleration), 99)) * 1.15)
    acceleration_ax.set_ylim(-acc_limit, acc_limit)
    acceleration_ax.set_title(f'加速度 {acceleration[ego_i]:.2f} m/s²')
    acceleration_ax.set_xlabel('时间 (s)')
    acceleration_ax.set_ylabel('加速度 (m/s²)')
    acceleration_ax.grid(alpha=0.2)

    if cot_ax is not None:
        cot_ax.clear()
        cot_ax.axis('off')
        elapsed_us = int(gif_timestamps[frame_number] - ego_ts[0])
        event_i = int(np.searchsorted(coc_start_us, elapsed_us, side='right') - 1)
        if event_i >= 0:
            cot_text = str(coc_events[event_i]['coc']).strip()
            wrapped_cot = textwrap.fill(cot_text, width=145)
            cot_ax.text(
                0.5, 0.5, f'CoT（CoC 标注）: {wrapped_cot}',
                ha='center', va='center', fontsize=13, color='#f8fafc',
                bbox={'boxstyle': 'round,pad=0.7', 'facecolor': '#312e81',
                      'edgecolor': '#818cf8', 'alpha': 0.95},
                transform=cot_ax.transAxes,
            )
        else:
            cot_ax.text(
                0.5, 0.5, 'CoT（CoC 标注）: 等待首个事件',
                ha='center', va='center', fontsize=12, color='#94a3b8',
                transform=cot_ax.transAxes,
            )

    fig.suptitle(f'PhysicalAI-AV {CLIP_ID[:8]}…  t={current_t:.1f}s')
    apply_cjk_font(fig)
    artists = [*camera_axes, trajectory_ax, speed_ax, acceleration_ax]
    return artists + ([cot_ax] if cot_ax is not None else [])

gif_path = OUTPUT_DIR / f'{CLIP_ID}.multiview_ego.gif'
clip_animation = animation.FuncAnimation(
    fig,
    draw_gif_frame,
    frames=n_gif_frames,
    interval=1000 / GIF_FPS,
    blit=False,
    repeat=True,
)
clip_animation.save(gif_path, writer=animation.PillowWriter(fps=GIF_FPS), dpi=90)
plt.close(fig)

summary = {
    'clip_id': CLIP_ID,
    'chunk': CHUNK_ID,
    'split': str(avdi.clip_index.loc[CLIP_ID, 'split']),
    'cache_dir': str(CACHE_DIR),
    'data_mode': 'streaming' if MAYBE_STREAM else 'local_cache',
    'download_features': [str(feature) for feature in DOWNLOAD_FEATURES],
    'ego_frames': int(len(ego)),
    'duration_s': float(ego['t_s'].iloc[-1]),
    'avg_speed_mps': float(ego['speed'].mean()),
    'max_speed_mps': float(ego['speed'].max()),
    'max_abs_acceleration_mps2': float(np.nanmax(np.abs(ego['acceleration']))),
    'gif_path': str(gif_path),
    'gif_fps': GIF_FPS,
    'gif_frames': n_gif_frames,
    'gif_camera_features': [str(feature) for feature, _ in camera_items],
    'obstacle_offline_available': obstacle_data is not None,
    'obstacle_offline_error': obstacle_load_error,
    'coc_available': bool(coc_events),
    'coc_event_count': len(coc_events),
    'coc_events': coc_events,
    'reasoning_load_error': reasoning_load_error,
}
(OUTPUT_DIR / f'{CLIP_ID}.summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
ego.to_csv(OUTPUT_DIR / f'{CLIP_ID}.egomotion.csv', index=False)
print('输出目录:', OUTPUT_DIR)
print('综合 GIF:', gif_path)
print(summary)
